# E7c — non-financial series (weekly CO2, monthly El Niño SST)

Accelerator: None, Internet: ON. About 30 minutes on CPU.


In [ ]:
# ============================================================================
# SEL 1. KONFIGURASI, ANGGARAN WAKTU, CACHE
# ============================================================================
import os, glob, math, time, pickle, random, warnings, shutil, json
import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

T_MULAI = time.time()
TIME_BUDGET_H = 11.3
RUN = dict(E7C=True, E7B=False, E7A=False)
OPTS_E7A = ('GA', 'PSO')
SEED = 42
FIT_EPOCHS, FINAL_EPOCHS, POP = 20, 100, 20
# E7b
ARCHS = ['lstm', 'gru', 'bilstm', 'tcn', 'transformer', 'mlp']
R_ARCH, BUDGET_ARCH, N_SEED_ARCH = 3, 100, 5
# E7a
R_REP, GENS, CHECK_REP, N_SEED_REP = 5, 15, (50, 100, 300), 5
# E7c
N_SEED_EXT = 5
SMOKE_TEST = False

OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else './out_e7'
CACHE = os.path.join(OUT, 'cache_e7'); os.makedirs(CACHE, exist_ok=True)
_n = 0
for f in (glob.glob('/kaggle/input/**/cache_e7/*.pkl', recursive=True) + glob.glob('/kaggle/input/**/cache_e6/*.pkl', recursive=True)):
    t = os.path.join(CACHE, os.path.basename(f))
    if not os.path.exists(t):
        try: shutil.copy(f, t); _n += 1
        except Exception: pass
print(f'cache dipulihkan: {_n} | total: {len(glob.glob(CACHE + "/*.pkl"))}')

def jam(): return (time.time() - T_MULAI) / 3600
def sisa(): return TIME_BUDGET_H - jam()
DILEWATI = []
def ada(k): return os.path.exists(os.path.join(CACHE, k + '.pkl'))
def disk(k, fn):
    p = os.path.join(CACHE, k + '.pkl')
    if os.path.exists(p):
        with open(p, 'rb') as f: return pickle.load(f)
    v = fn(); tmp = p + f'.tmp{os.getpid()}'
    with open(tmp, 'wb') as f: pickle.dump(v, f)
    os.replace(tmp, p); return v
def skip(tag):
    if tag not in DILEWATI: DILEWATI.append(tag)
    print(f'[jeda] {tag}: anggaran waktu commit habis, lanjut di commit berikutnya')

if SMOKE_TEST:
    R_ARCH, BUDGET_ARCH, N_SEED_ARCH = 1, 6, 2; R_REP, GENS, CHECK_REP, N_SEED_REP = 1, 3, (6, 12), 2
    N_SEED_EXT = 2; FIT_EPOCHS, FINAL_EPOCHS, POP = 1, 2, 4

mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 600})

In [ ]:
# ============================================================================
# SEL 2. DATA SIDO, MODEL SEMUA ARSITEKTUR, PELATIHAN (protokol naskah)
# ============================================================================
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers as KL
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
print('Perangkat:', tf.config.list_physical_devices('GPU') or f'CPU ({os.cpu_count()} core)')

def set_seed(s): random.seed(s); np.random.seed(s); tf.random.set_seed(s)
TARGET, TRAIN_FRAC = 'Close', 0.8
START, END = '2019-07-01', '2025-07-01'

# --- pengunduh Yahoo yang tahan perubahan versi -------------------------------------------
# Image Kaggle kadang memasang yfinance yang meminta impersonasi "chrome150" padahal curl_cffi-nya lebih lama
# ("Impersonating chrome150 is not supported"). Urutan perbaikan: (a) perbarui curl_cffi + yfinance,
# (b) beri yfinance sesi curl_cffi dengan profil yang didukung, (c) endpoint chart Yahoo langsung via requests.
import subprocess, sys
if not SMOKE_TEST:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'curl_cffi', 'yfinance'], check=False, timeout=300,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception as ex:
        print('pip upgrade dilewati:', ex)

def _session():
    try:
        from curl_cffi import requests as creq
    except Exception:
        return None
    for imp in ('chrome', 'chrome131', 'chrome124', 'chrome120', 'chrome116', 'chrome110', 'safari', 'edge101'):
        try:
            return creq.Session(impersonate=imp)
        except Exception:
            continue
    return None
SESSION = None if SMOKE_TEST else _session()

def _chart_api(tk):
    """Endpoint chart Yahoo v8 langsung (tanpa yfinance). Harga disesuaikan dividen/split (adjclose) seperti auto_adjust."""
    import requests
    p1 = int(pd.Timestamp(START).timestamp()); p2 = int(pd.Timestamp(END).timestamp())
    for host in ('query1', 'query2'):
        r = requests.get(f'https://{host}.finance.yahoo.com/v8/finance/chart/{tk}',
                         params=dict(period1=p1, period2=p2, interval='1d', events='div,splits'),
                         headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124 Safari/537.36'},
                         timeout=30)
        if r.status_code != 200: continue
        res = r.json()['chart']['result'][0]; q = res['indicators']['quote'][0]
        adj = res['indicators'].get('adjclose', [{}])[0].get('adjclose', q['close'])
        d = pd.DataFrame({'Date': pd.to_datetime(res['timestamp'], unit='s'), 'Open': q['open'], 'High': q['high'],
                          'Low': q['low'], 'Close': q['close'], 'Volume': q['volume'], 'Adj': adj}).dropna()
        f = d['Adj'] / d['Close']                       # faktor penyesuaian, setara auto_adjust=True
        for c in ('Open', 'High', 'Low', 'Close'): d[c] = d[c] * f
        d['Date'] = (d['Date'] + pd.Timedelta(hours=7)).dt.normalize()   # zona WIB/pasar Asia
        return d.drop(columns='Adj')
    raise RuntimeError(f'chart API status {r.status_code}')

def _yf(tk, cols=('Open', 'High', 'Low', 'Close', 'Volume'), tries=6):
    """Coba bergantian: yfinance.download (dengan sesi), yfinance.Ticker.history, endpoint chart langsung."""
    import yfinance as yf
    last = None
    for i in range(tries):
        try:
            m = i % 3
            if m == 0:
                d = yf.download(tk, start=START, end=END, auto_adjust=True, progress=False, threads=False,
                                **({'session': SESSION} if SESSION is not None else {}))
            elif m == 1:
                d = (yf.Ticker(tk, session=SESSION) if SESSION is not None else yf.Ticker(tk)).history(start=START, end=END, auto_adjust=True)
            else:
                d = _chart_api(tk)
            if isinstance(d.columns, pd.MultiIndex): d.columns = d.columns.get_level_values(0)
            if 'Date' not in d.columns:
                d = d.reset_index()
                if 'Date' not in d.columns: d = d.rename(columns={d.columns[0]: 'Date'})
            d['Date'] = pd.to_datetime(d['Date'])
            if getattr(d['Date'].dt, 'tz', None) is not None: d['Date'] = d['Date'].dt.tz_localize(None)
            d['Date'] = d['Date'].dt.normalize()
            if len(d) > 200:
                print(f'  {tk}: {len(d)} baris (metode {["yf.download", "yf.Ticker", "chart API"][m]})')
                return d[['Date', *cols]].dropna().reset_index(drop=True)
            last = f'{len(d)} baris'
        except Exception as ex:
            last = repr(ex)[:160]
        tunggu = 10*(i+1) + random.random()*10
        print(f'  {tk}: percobaan {i+1} gagal ({last}); tunggu {tunggu:.0f} detik')
        time.sleep(tunggu)
    raise RuntimeError(f'semua metode unduh gagal untuk {tk}: {last}')

def _ema(s, n): return s.ewm(span=n, adjust=False).mean()
def add_technical(df):
    d = df.copy(); c = d.Close
    d['SMA10'] = c.rolling(10).mean(); d['SMA20'] = c.rolling(20).mean(); d['EMA10'] = _ema(c, 10); d['EMA20'] = _ema(c, 20)
    dl = c.diff(); up = dl.clip(lower=0); dn = -dl.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / dn.ewm(alpha=1/14, adjust=False).mean(); d['RSI14'] = 100 - 100/(1+rs)
    m = _ema(c, 12) - _ema(c, 26); d['MACD'] = m; d['MACD_sig'] = _ema(m, 9)
    d['ret1'] = c.pct_change(); d['vol10'] = d['ret1'].rolling(10).std(); return d

def load_sido():
    """Urutan sumber: (1) cache data dari commit sebelumnya/notebook E7 lain, (2) Yahoo (dengan coba ulang),
       (3) berkas SIDO_E7_data.csv yang diunggah sebagai dataset. Data yang berhasil dimuat disimpan ke cache
       dan ke OUT/SIDO_E7_data.csv, supaya commit berikutnya tidak perlu Yahoo lagi."""
    if not SMOKE_TEST:
        if ada('data_sido_e7'): d = disk('data_sido_e7', None); print(f'data SIDO dari cache: {len(d)} baris'); return d
        try:
            d = _load_sido_raw()
        except Exception as ex:
            f = glob.glob('/kaggle/input/**/SIDO_E7_data.csv', recursive=True)
            if not f:
                print('GAGAL MEMUAT DATA SIDO'); raise SystemExit(f'Data SIDO tidak bisa dimuat ({ex}).\n'
                                 '  1) Pastikan Internet ON (panel kanan → Settings → Internet).\n'
                                 '  2) Jalankan notebook E7 lain lebih dulu, lalu Add Input output-nya (berisi cache_e7/data_sido_e7.pkl\n'
                                 '     atau SIDO_E7_data.csv), atau tunggu 10–15 menit lalu commit ulang (batas permintaan Yahoo).')
            d = pd.read_csv(f[0], parse_dates=['Date']); print('data SIDO dari', f[0])
            if 'SMA10' not in d.columns:                      # CSV mentah OHLCV (+ JKSE, USDIDR opsional)
                need = {'Open', 'High', 'Low', 'Close', 'Volume'} - set(d.columns)
                if need: raise SystemExit(f'SIDO_E7_data.csv kurang kolom {need}; dibutuhkan Date, Open, High, Low, Close, Volume')
                ext = {c: d[c] for c in ('JKSE', 'USDIDR') if c in d.columns}
                d = add_technical(d[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']])
                for c in ('JKSE', 'USDIDR'): d[c] = ext[c].ffill().bfill() if c in ext else 0.0
                d = d.dropna().reset_index(drop=True)
        if len(d) < 1000: raise SystemExit(f'Data SIDO hanya {len(d)} baris; periksa sumber data.')
        disk('data_sido_e7', lambda: d); d.to_csv(f'{OUT}/SIDO_E7_data.csv', index=False)
        return d
    return _load_sido_raw()

def _load_sido_raw():
    if SMOKE_TEST:
        r = np.random.RandomState(0); n = 1449; c = 600*np.exp(np.cumsum(r.normal(0, .015, n)))
        raw = pd.DataFrame({'Date': pd.date_range(START, periods=n, freq='B'), 'Open': c, 'High': c*1.01, 'Low': c*.99,
                            'Close': c, 'Volume': r.randint(1e5, 1e6, n).astype(float)})
    else:
        raw = _yf('SIDO.JK')
    d = add_technical(raw)
    for name, tk in {'JKSE': '^JKSE', 'USDIDR': 'USDIDR=X'}.items():
        try:
            if SMOKE_TEST: raise RuntimeError
            e = _yf(tk, cols=('Close',)).rename(columns={'Close': name}); d = d.merge(e, on='Date', how='left'); d[name] = d[name].ffill().bfill()
        except Exception:
            d[name] = 0.0; print(f'PERINGATAN: {name} diisi nol')
    return d.dropna().reset_index(drop=True)

df_sido = load_sido()
FEATS = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA10', 'SMA20', 'EMA10', 'EMA20', 'RSI14', 'MACD', 'MACD_sig', 'ret1', 'vol10', 'JKSE', 'USDIDR']
n_tr = int(len(df_sido)*TRAIN_FRAC); y_all = df_sido.Close.values.astype(float)
y_te, rw_te = y_all[n_tr:], y_all[n_tr-1:-1]
def mape(y, p): return float(np.mean(np.abs((y-p)/y))*100)
def mase(y, p, ytr): return float(np.mean(np.abs(y-p)) / np.mean(np.abs(np.diff(ytr))))
NAIVE = mape(y_te, rw_te); print(f'SIDO: {len(df_sido)} baris | MAPE random walk {NAIVE:.3f}')

OPTS = ['adam', 'rmsprop', 'nadam']
def _opt(name, lr): return {'adam': Adam, 'rmsprop': RMSprop, 'nadam': Nadam}[name](lr)
class LastStep(KL.Layer):
    def call(self, x): return x[:, -1, :]
class PosEnc(KL.Layer):
    def build(self, shape):
        T, D = int(shape[1]), int(shape[2]); pos = np.arange(T)[:, None]; i = np.arange(D)[None, :]
        ang = pos / np.power(10000, (2*(i//2))/D)
        self.pe = tf.constant(np.where(i % 2 == 0, np.sin(ang), np.cos(ang))[None], dtype='float32')
    def call(self, x): return x + self.pe

def build_model(shape, hp, arch='lstm'):
    """n_layers berarti: lapisan rekuren (LSTM/GRU/BiLSTM), blok konvolusi berdilatasi (TCN: 1+n_layers lapis,
       dilasi 1,2,4,...), blok encoder (Transformer), atau lapisan tersembunyi (MLP)."""
    K.clear_session(); inp = KL.Input(shape=shape); u = int(hp['units']); dr = float(hp['dropout']); L = int(hp['n_layers'])
    if arch in ('lstm', 'gru', 'bilstm'):
        x = inp
        for i in range(L):
            cell = (KL.GRU if arch == 'gru' else KL.LSTM)(u, return_sequences=(i < L-1))
            x = KL.Bidirectional(cell)(x) if arch == 'bilstm' else cell(x)
            if dr > 0: x = KL.Dropout(dr)(x)
    elif arch == 'tcn':
        x = inp
        for j in range(L + 1):
            x = KL.Conv1D(u, 3, padding='causal', dilation_rate=2**j, activation='relu')(x)
            if dr > 0: x = KL.Dropout(dr)(x)
        x = LastStep()(x)
    elif arch == 'mlp':
        x = KL.Flatten()(inp)
        for _ in range(L):
            x = KL.Dense(u, activation='relu')(x)
            if dr > 0: x = KL.Dropout(dr)(x)
    elif arch == 'transformer':
        x = PosEnc()(KL.Dense(u)(inp))
        for _ in range(L):
            a = KL.MultiHeadAttention(num_heads=4, key_dim=max(1, u//4), dropout=dr)(x, x); x = KL.LayerNormalization()(x + a)
            f = KL.Dense(u, activation='relu')(x); f = KL.Dense(u)(f); x = KL.LayerNormalization()(x + f)
        x = KL.GlobalAveragePooling1D()(x)
        if dr > 0: x = KL.Dropout(dr)(x)
    m = Model(inp, KL.Dense(1, activation='linear')(x))
    m.compile(loss='mse', optimizer=_opt(hp['optimizer'], hp['lr'])); return m

def prepare(df, feats, ts):
    dX = df[feats].values.astype('float32'); dy = df[[TARGET]].values.astype('float32')
    ntr = int(len(df)*TRAIN_FRAC)
    sx = MinMaxScaler().fit(dX[:ntr]); sy = MinMaxScaler().fit(dy[:ntr])
    Xs, ys = sx.transform(dX), sy.transform(dy).ravel()
    X = np.stack([Xs[i-ts:i] for i in range(ts, len(df))]); Y = ys[ts:]; sp = ntr - ts
    return X[:sp], Y[:sp], X[sp:], sy

def train_pred(hp, seed, arch='lstm', df=None, feats=None):
    df = df_sido if df is None else df; feats = FEATS if feats is None else feats
    set_seed(seed); Xtr, Ytr, Xte, sy = prepare(df, feats, int(hp['timestep']))
    m = build_model(Xtr.shape[1:], hp, arch)
    m.fit(Xtr, Ytr, validation_split=0.2, shuffle=False, epochs=FINAL_EPOCHS, batch_size=int(hp['batch_size']),
          callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
    pred = sy.inverse_transform(m.predict(Xte, verbose=0).reshape(-1, 1)).ravel()
    y = df[TARGET].values.astype(float); ntr = int(len(df)*TRAIN_FRAC)
    return dict(pred=pred, y=y[ntr:], rw=y[ntr-1:-1])

def dm_test(e1, e2, h=1):
    d = np.abs(e1) - np.abs(e2); n = len(d); db = d.mean()
    g = [np.sum((d[k:]-db)*(d[:n-k]-db))/n for k in range(h)]
    dm = db/math.sqrt((g[0] + 2*sum(g[1:]))/n) * math.sqrt((n + 1 - 2*h + h*(h-1)/n)/n)
    return float(dm), float(2*stats.t.sf(abs(dm), df=n-1))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); a = np.empty(m); r = 0.0
    for k, i in enumerate(o): r = max(r, min(1.0, (m-k)*p[i])); a[i] = r
    return a

In [ ]:
# ============================================================================
# SEL 3. RUANG PENCARIAN, FITNESS (shared / nested), GA & PSO
# ============================================================================
SPACE = dict(units=(16, 160), n_layers=(1, 3), dropout=(0.10, 0.50), lr=(1e-4, 1e-3), batch_size=(16, 128), timestep=(10, 30))
LB = np.array([16, 1, 0.10, -4.0, 16, 10, 0], float)
UB = np.array([160, 3, 0.50, -3.0, 128, 30, len(OPTS) - 1e-9], float)
def vec_to_hp(v):
    return dict(units=int(round(v[0])), n_layers=int(round(v[1])), dropout=round(float(v[2]), 3), lr=float(10**v[3]),
                batch_size=int(round(v[4])), timestep=int(round(v[5])), optimizer=OPTS[int(min(int(v[6]), len(OPTS)-1))])
def cfg_key(v):
    h = vec_to_hp(v)
    return (h['units'], h['n_layers'], round(h['dropout'], 2), round(math.log10(h['lr']), 2), h['batch_size'], h['timestep'], h['optimizer'])
def key_to_hp(k):
    return dict(units=int(k[0]), n_layers=int(k[1]), dropout=float(k[2]), lr=float(10**k[3]), batch_size=int(k[4]), timestep=int(k[5]), optimizer=k[6])
def ks(k): return '_'.join(map(str, k))
def on_bounds(hp):
    out = []
    for kk in ('units', 'dropout', 'lr', 'batch_size', 'timestep'):
        lo, hi, v = SPACE[kk][0], SPACE[kk][1], float(hp[kk])
        if kk == 'lr': lo, hi, v = math.log10(lo), math.log10(hi), math.log10(v)
        if min(abs(v-lo), abs(v-hi)) <= 0.01*(hi-lo): out.append(kk)
    return ','.join(out) or 'none'

class Habis(Exception): pass
def fitness(k, mode, arch):
    """shared: latih 0-80% porsi latih, nilai 80-100% (blok yang juga dipakai early stopping) = protokol naskah.
       nested: latih 0-60%, nilai 60-80%; blok 80-100% hanya untuk early stopping final."""
    key = {('shared', 'lstm'): 'fit_', ('nested', 'lstm'): 'fitN_'}.get((mode, arch), f'fit7_{mode}_{arch}_') + ks(k)
    if not ada(key) and sisa() < 0.4: raise Habis()
    hp = key_to_hp(k)
    def _hit():
        set_seed(SEED); Xtr, Ytr, _, _ = prepare(df_sido, FEATS, hp['timestep']); n = len(Xtr); b = int(0.2*n)
        if mode == 'shared': Xa, ya, Xv, yv = Xtr[:n-b], Ytr[:n-b], Xtr[n-b:], Ytr[n-b:]
        else: Xa, ya, Xv, yv = Xtr[:n-2*b], Ytr[:n-2*b], Xtr[n-2*b:n-b], Ytr[n-2*b:n-b]
        m = build_model(Xa.shape[1:], hp, arch)
        m.fit(Xa, ya, epochs=FIT_EPOCHS, shuffle=False, batch_size=hp['batch_size'], verbose=0)
        return float(np.mean((m.predict(Xv, verbose=0).ravel() - yv.ravel())**2))
    return disk(key, _hit)

class Evaluator:
    """count='distinct': anggaran = kandidat berbeda (E6/E7b). count='total': anggaran = semua evaluasi,
       termasuk kandidat berulang (protokol naskah: populasi 20 x 15 generasi = 300)."""
    def __init__(self, mode, arch): self.memo, self.hist, self.mode, self.arch, self.calls = {}, [], mode, arch, 0
    def __call__(self, v):
        k = cfg_key(v); self.calls += 1
        if k not in self.memo: self.memo[k] = fitness(k, self.mode, self.arch)
        bk = min(self.memo, key=self.memo.get); self.hist.append((self.calls, len(self.memo), self.memo[bk], bk))
        return self.memo[k]

def ga_e6(seed, mode, arch, budget):          # operator E6 (uniform crossover, mutasi Gaussian), anggaran distinct
    rng = np.random.RandomState(seed); F = Evaluator(mode, arch)
    X = rng.uniform(LB, UB, size=(POP, 7)); f = np.array([F(x) for x in X]); stall = 0
    while len(F.memo) < budget and stall < 50:
        before = len(F.memo); order = np.argsort(f); newX = [X[i].copy() for i in order[:POP//4]]
        while len(newX) < POP:
            a, b = [X[rng.choice(order[:POP//2])] for _ in range(2)]
            ch = np.where(rng.rand(7) < 0.5, a, b); mut = rng.rand(7) < 0.2
            newX.append(np.clip(np.where(mut, ch + rng.normal(0, 0.1, 7)*(UB-LB), ch), LB, UB))
        X = np.array(newX); f = np.array([F(x) for x in X]); stall = stall + 1 if len(F.memo) == before else 0
    return F

def ga_paper(seed, gens=GENS):                 # operator naskah: elit 1/4, blend crossover a=0.25, mutasi seragam p=0.3
    rng = np.random.RandomState(seed); F = Evaluator('shared', 'lstm')
    X = rng.uniform(LB, UB, size=(POP, 7)); f = np.array([F(x) for x in X])
    for g in range(gens - 1):
        order = np.argsort(f); newX = [X[i].copy() for i in order[:POP//4]]
        while len(newX) < POP:
            p1, p2 = [X[rng.choice(order[:POP//2])] for _ in range(2)]
            beta = rng.uniform(-0.25, 1.25, 7); ch = p1 + beta*(p2 - p1)
            mut = rng.rand(7) < 0.3; ch = np.where(mut, rng.uniform(LB, UB), ch)
            newX.append(np.clip(ch, LB, UB))
        X = np.array(newX); f = np.array([F(x) for x in X])
    return F

def pso_paper(seed, iters=GENS, w=0.729, c1=1.49445, c2=1.49445):   # PSO konstriksi naskah (Alg. 2), tanpa re-inisialisasi
    rng = np.random.RandomState(seed); F = Evaluator('shared', 'lstm')
    X = rng.uniform(LB, UB, size=(POP, 7)); V = np.zeros_like(X)
    pbf = np.array([F(x) for x in X]); pb = X.copy(); gb = X[pbf.argmin()].copy(); gbf = pbf.min()
    for t in range(iters - 1):
        r1, r2 = rng.rand(*X.shape), rng.rand(*X.shape)
        V = w*V + c1*r1*(pb - X) + c2*r2*(gb - X); X = np.clip(X + V, LB, UB)
        for i in range(POP):
            f = F(X[i])
            if f < pbf[i]: pbf[i], pb[i] = f, X[i].copy()
            if f < gbf: gbf, gb = f, X[i].copy()
    return F

def run_search(tag, fn):
    if ada(tag): return disk(tag, None)
    try: F = fn()
    except Habis: skip(tag); return None
    return disk(tag, lambda: dict(hist=F.hist, n_distinct=len(F.memo), calls=F.calls))

def incumbent(res, b, by='distinct'):
    col = 1 if by == 'distinct' else 0
    rows = [h for h in res['hist'] if h[col] <= b]
    return (rows[-1] if rows else res['hist'][-1])[3], (rows[-1] if rows else res['hist'][-1])[2]

def final_eval(k, arch, n_seed):
    hp = key_to_hp(k); outs = []
    for s in range(n_seed):
        key = (f'fin_{ks(k)}_s{s}' if arch == 'lstm' else f'fin7_{arch}_{ks(k)}_s{s}')
        if not ada(key) and sisa() < 0.3: skip(key); break
        outs.append(disk(key, lambda s=s: train_pred(hp, SEED + s, arch)))
    return outs
print('siap')

In [ ]:
# ============================================================================
# SEL 4. E7c — deret non-keuangan dan frekuensi lain (Reviewer A11)
#   CO2 Mauna Loa mingguan (statsmodels, 1958-2001) dan SST El Nino bulanan (statsmodels, 1950-2010).
#   Model: LSTM interior (Close saja, 5 seed) dan AR linear pada selisih (p = periode musiman) sebagai
#   pembanding yang memang memanfaatkan struktur musiman. DM vs random walk, Holm antar-uji.
# ============================================================================
INT_HP = dict(units=88, n_layers=1, dropout=0.3, lr=3.2e-4, batch_size=64, timestep=20, optimizer='adam')
def load_external():
    if SMOKE_TEST:
        t = np.arange(2000); return {'CO2 (weekly)': (pd.DataFrame({'Close': 320 + .03*t + 3*np.sin(2*np.pi*t/52)}), 52),
                                     'El Niño SST (monthly)': (pd.DataFrame({'Close': 25 + 1.5*np.sin(2*np.pi*np.arange(700)/12)}), 12)}
    import statsmodels.api as sm
    co2 = sm.datasets.co2.load_pandas().data['co2'].interpolate().dropna().values
    el = sm.datasets.elnino.load_pandas().data.set_index('YEAR').stack().values.astype(float)
    return {'CO2 (weekly)': (pd.DataFrame({'Close': co2}), 52), 'El Niño SST (monthly)': (pd.DataFrame({'Close': el}), 12)}

def ar_diff_forecast(y, ntr, p):
    d = np.diff(y); X = np.array([d[i-p:i][::-1] for i in range(p, len(d))]); Y = d[p:]
    idx = np.arange(p, len(d)) + 1                     # indeks y yang diramal: y[idx] = y[idx-1] + d[idx-1]
    tr = idx < ntr; A = np.c_[X, np.ones(len(X))]; w = np.linalg.lstsq(A[tr], Y[tr], rcond=None)[0]
    te = idx >= ntr; return y[idx[te]-1] + A[te] @ w

E7C = pd.DataFrame()
if RUN['E7C']:
    rows = []
    for name, (df, p) in load_external().items():
        y = df.Close.values.astype(float); ntr = int(len(y)*TRAIN_FRAC); yt, rw = y[ntr:], y[ntr-1:-1]
        outs = []
        for s in range(N_SEED_EXT):
            key = f'ext7_{name.split()[0]}_{s}'
            if not ada(key) and sisa() < 0.3: skip(key); break
            outs.append(disk(key, lambda s=s: train_pred(INT_HP, SEED + s, 'lstm', df=df, feats=['Close'])))
        cand = {'AR on differences (p=%d)' % p: ar_diff_forecast(y, ntr, p)}
        if outs: cand['LSTM (interior, %d seeds)' % len(outs)] = np.mean([o['pred'] for o in outs], 0)
        for mdl, pr in cand.items():
            dm, pv = dm_test(yt - pr, yt - rw)
            rows.append(dict(series=name, n_obs=len(y), model=mdl, MAPE_model=mape(yt, pr), MAPE_RW=mape(yt, rw),
                             MASE=mase(yt, pr, y[:ntr]), DM=dm, p_raw=pv,
                             share_test_outside_train=float(np.mean((yt > y[:ntr].max()) | (yt < y[:ntr].min())))))
    E7C = pd.DataFrame(rows)
    if len(E7C):
        E7C['p_holm'] = holm(E7C.p_raw)
        E7C['verdict'] = np.where(E7C.p_holm >= .05, 'n.s.', np.where(E7C.DM < 0, 'model better', 'RW better'))
        E7C.round(4).to_csv(f'{OUT}/TableE7c_nonfinancial.csv', index=False); print(E7C.round(3).to_string(index=False))

In [ ]:
# ============================================================================
# SEL 5. E7b — perbandingan arsitektur yang adil + seleksi bersarang (Reviewer B3, B4)
#   Setiap arsitektur: GA yang sama, ruang asli yang sama, 100 evaluasi berbeda, R_ARCH run (seed 42, 1042, 2042),
#   fitness pada blok yang terpisah dari blok early stopping. Konfigurasi terbaik tiap run -> N_SEED_ARCH seed.
#   DM pada prakiraan rata-rata seed dari run terbaik-validasi tiap arsitektur, Holm antar-arsitektur.
# ============================================================================
E7B = pd.DataFrame(); E7B_DM = pd.DataFrame()
if RUN['E7B']:
    rows = []; preds = {}
    for arch in ARCHS:
        for r in range(R_ARCH):
            tag = f'srch7b_{arch}_r{r}'
            res = run_search(tag, lambda arch=arch, r=r: ga_e6(SEED + 1000*r, 'nested', arch, BUDGET_ARCH))
            if res is None: continue
            k, vf = incumbent(res, BUDGET_ARCH, 'distinct'); outs = final_eval(k, arch, N_SEED_ARCH)
            if not outs: continue
            a = np.array([mape(o['y'], o['pred']) for o in outs]); hp = key_to_hp(k)
            rows.append(dict(arch=arch, run=r, best_val_mse=vf, n_seed=len(a), test_MAPE_mean=a.mean(),
                             test_MAPE_sd=a.std(ddof=1) if len(a) > 1 else np.nan, on_bounds=on_bounds(hp),
                             **{f'hp_{kk}': vv for kk, vv in hp.items()}))
            preds[(arch, r)] = (vf, np.mean([o['pred'] for o in outs], 0))
            print(f'  {tag}: val {vf:.6f} | uji {a.mean():.3f}% | {hp}')
    E7B = pd.DataFrame(rows)
    if len(E7B):
        E7B.round(5).to_csv(f'{OUT}/TableE7b_tuned_archs_runs.csv', index=False)
        S = (E7B.groupby('arch').agg(runs=('run', 'count'), test_MAPE_mean=('test_MAPE_mean', 'mean'), sd_runs=('test_MAPE_mean', 'std'),
                                     test_MAPE_min=('test_MAPE_mean', 'min'), test_MAPE_max=('test_MAPE_mean', 'max'))
               .reset_index().sort_values('test_MAPE_mean'))
        dms = []
        for arch in S.arch:
            cands = [(v, p) for (a_, r), (v, p) in preds.items() if a_ == arch]
            v, p = min(cands, key=lambda t: t[0])          # run yang dipilih dengan kriteria validasi, bukan uji
            dm, pv = dm_test(y_te - p, y_te - rw_te); dms.append(dict(arch=arch, MAPE_selected=mape(y_te, p), DM=dm, p_raw=pv))
        D = pd.DataFrame(dms); D['p_holm'] = holm(D.p_raw)
        D['verdict'] = np.where(D.p_holm >= .05, 'n.s.', np.where(D.DM > 0, 'RW better', 'model better'))
        S = S.merge(D, on='arch'); S['vs_naive'] = S.test_MAPE_mean / NAIVE
        if S.runs.min() >= 2 and S.arch.nunique() > 2:
            S['kruskal_p_all_archs'] = stats.kruskal(*[g.test_MAPE_mean.values for _, g in E7B.groupby('arch')]).pvalue
        S.round(4).to_csv(f'{OUT}/TableE7b_tuned_archs_summary.csv', index=False); print('\n' + S.round(3).to_string(index=False))
        print(f'RW = {NAIVE:.3f}. DM > 0: random walk lebih akurat.')

In [ ]:
# ============================================================================
# SEL 6. E7a — replikasi persis protokol pencarian naskah (Reviewer B2)
#   GA & PSO dengan operator naskah, 300 evaluasi (20 x 15, kandidat berulang ikut dihitung seperti di naskah),
#   ruang asli, validasi bersama (seperti naskah), R_REP run independen. MAPE uji di anggaran 50, 100, 300.
# ============================================================================
E7A = pd.DataFrame()
if RUN['E7A']:
    rows = []
    for opt in OPTS_E7A:
        for r in range(R_REP):
            tag = f'srch7a_{opt}_r{r}'
            res = run_search(tag, lambda opt=opt, r=r: (ga_paper if opt == 'GA' else pso_paper)(SEED + 1000*r))
            if res is None: continue
            for b in CHECK_REP:
                k, vf = incumbent(res, b, 'total'); outs = final_eval(k, 'lstm', N_SEED_REP)
                if not outs: continue
                a = np.array([mape(o['y'], o['pred']) for o in outs]); hp = key_to_hp(k)
                rows.append(dict(optimizer=opt, run=r, budget=b, distinct_at_budget=max(h[1] for h in res['hist'] if h[0] <= b),
                                 best_val_mse=vf, n_seed=len(a), test_MAPE_mean=a.mean(), test_MAPE_sd=a.std(ddof=1) if len(a) > 1 else np.nan,
                                 on_bounds=on_bounds(hp), **{f'hp_{kk}': vv for kk, vv in hp.items()}))
            print(f'  {tag}: {res["calls"]} evaluasi ({res["n_distinct"]} berbeda) | uji@{CHECK_REP[-1]} = '
                  f'{rows[-1]["test_MAPE_mean"]:.3f}%' if rows and rows[-1]['run'] == r and rows[-1]['optimizer'] == opt else '')
    E7A = pd.DataFrame(rows)
    if len(E7A):
        E7A.round(5).to_csv(f'{OUT}/TableE7a_paper_protocol_runs.csv', index=False)
        fin = E7A[E7A.budget == CHECK_REP[-1]]
        S = fin.groupby('optimizer').agg(runs=('run', 'count'), test_MAPE_mean=('test_MAPE_mean', 'mean'), sd_runs=('test_MAPE_mean', 'std'),
                                         test_MAPE_min=('test_MAPE_mean', 'min'), test_MAPE_max=('test_MAPE_mean', 'max'),
                                         runs_on_bound=('on_bounds', lambda s: int((s != 'none').sum())),
                                         distinct_evals=('distinct_at_budget', 'mean')).reset_index()
        g, q = fin[fin.optimizer == 'GA'].test_MAPE_mean, fin[fin.optimizer == 'PSO'].test_MAPE_mean
        if len(g) > 1 and len(q) > 1: S['GA_vs_PSO_mannwhitney_p'] = stats.mannwhitneyu(g, q, alternative='two-sided').pvalue
        S.round(4).to_csv(f'{OUT}/TableE7a_paper_protocol_summary.csv', index=False); print('\n' + S.round(3).to_string(index=False))
        B = E7A.groupby(['optimizer', 'budget']).agg(test=('test_MAPE_mean', 'mean'), val=('best_val_mse', 'median')).round(5)
        B.to_csv(f'{OUT}/TableE7a_budget.csv'); print('\n' + B.to_string())
        pv = E7A.pivot_table(index=['optimizer', 'run'], columns='budget', values='test_MAPE_mean').dropna()
        if len(pv) >= 5 and CHECK_REP[0] in pv and CHECK_REP[-1] in pv:
            print('Wilcoxon uji @%d vs @%d: p = %.3f' % (CHECK_REP[0], CHECK_REP[-1], stats.wilcoxon(pv[CHECK_REP[0]], pv[CHECK_REP[-1]]).pvalue))

        fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.1, 2.6))
        for opt, ls in (('GA', '-'), ('PSO', '--')):
            H = [disk(f'srch7a_{opt}_r{r}', None)['hist'] for r in range(R_REP) if ada(f'srch7a_{opt}_r{r}')]
            if not H: continue
            L = min(len(h) for h in H); M = np.array([[x[2] for x in h[:L]] for h in H]); x = np.arange(1, L+1)
            a1.plot(x, np.median(M, 0), ls, color='k', lw=1.2, label=f'{opt} (median, {len(H)} runs)')
            a1.fill_between(x, M.min(0), M.max(0), color='0.8', lw=0)
            m = E7A[E7A.optimizer == opt].groupby('budget').test_MAPE_mean.agg(['mean', 'min', 'max'])
            a2.errorbar(m.index + (0 if opt == 'GA' else 4), m['mean'], yerr=[m['mean']-m['min'], m['max']-m['mean']],
                        fmt='o' + ls, color='k' if opt == 'GA' else '0.5', ms=4, lw=1, capsize=2, label=opt)
        a1.set_xlabel('Fitness evaluations (paper protocol)'); a1.set_ylabel('Best validation MSE')
        a1.ticklabel_format(axis='y', style='sci', scilimits=(0, 0)); a1.legend(frameon=False, fontsize=7); a1.set_title('(a) Search criterion', loc='left')
        a2.axhline(NAIVE, color='k', lw=1); a2.text(.02, NAIVE, f'Random walk ({NAIVE:.2f})', transform=a2.get_yaxis_transform(), va='bottom', fontsize=6.5)
        a2.set_xlabel('Evaluation budget'); a2.set_ylabel('Test MAPE (%)'); a2.legend(frameon=False, fontsize=7); a2.set_title('(b) Test MAPE vs budget', loc='left')
        fig.tight_layout(); fig.savefig(f'{OUT}/FigE7a_paper_protocol.png', bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================================
# SEL 7. RINGKASAN
# ============================================================================
print(f'\nTerpakai {jam():.2f} jam dari {TIME_BUDGET_H}')
print('Dilewati:', DILEWATI[:6], '...' if len(DILEWATI) > 6 else '') if DILEWATI else print('Dilewati: tidak ada — E7 lengkap')
print('Isi cache:', len(glob.glob(CACHE + '/*.pkl')))